# 19. 3D point-cloud and detection — PointNet++, DGCNN, Point Transformer, PointPillars, CenterPoint

PointNet++ set abstraction, DGCNN EdgeConv, Point Transformer vector attention, PointPillars PFN과 CenterPoint의 center-based detection objective/decode를 작은 point count와 channel width로 구현한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")


## 1. PointNet++: FPS + radius grouping + PointNet with BatchNorm


In [ ]:
def farthest_point_sampling(xyz, count):
    point_count = xyz.size(0)
    selected = torch.zeros(count, dtype=torch.long, device=xyz.device)
    minimum_distance = torch.full((point_count,), float("inf"), device=xyz.device)
    farthest = torch.tensor(0, device=xyz.device)
    for sample_index in range(count):
        selected[sample_index] = farthest
        centroid = xyz[farthest : farthest + 1]
        distance = (xyz - centroid).square().sum(dim=-1)
        minimum_distance = torch.minimum(minimum_distance, distance)
        farthest = minimum_distance.argmax()
    return selected

def ball_query(query_xyz, source_xyz, radius, samples):
    distances = torch.cdist(query_xyz, source_xyz)
    groups = []
    for distance_row in distances:
        valid = torch.where(distance_row <= radius)[0]
        if valid.numel() == 0:
            valid = distance_row.argmin().view(1)
        chosen = valid[:samples]
        if chosen.numel() < samples:
            chosen = torch.cat([chosen, chosen[-1:].repeat(samples - chosen.numel())])
        groups.append(chosen)
    return torch.stack(groups)

class SharedPointMLP(nn.Module):
    def __init__(self, input_dim, mlp_dims):
        super().__init__()
        self.layers = nn.ModuleList()
        current = input_dim
        for output in mlp_dims:
            self.layers.append(nn.Sequential(nn.Linear(current, output, bias=False), nn.BatchNorm1d(output), nn.ReLU()))
            current = output
    def forward(self, grouped):
        shape = grouped.shape
        hidden = grouped.reshape(-1, shape[-1])
        for layer in self.layers:
            hidden = layer(hidden)
        return hidden.view(*shape[:-1], -1)

class SetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, mlp_dims):
        super().__init__()
        self.local_pointnet = SharedPointMLP(3 + input_feature_dim, mlp_dims)
    def forward(self, source_xyz, source_features, centroid_count, radius, neighbors):
        centroid_ids = farthest_point_sampling(source_xyz, centroid_count)
        centroid_xyz = source_xyz[centroid_ids]
        neighbor_ids = ball_query(centroid_xyz, source_xyz, radius, neighbors)
        relative_xyz = source_xyz[neighbor_ids] - centroid_xyz[:, None]
        local_input = relative_xyz if source_features is None else torch.cat([relative_xyz, source_features[neighbor_ids]], dim=-1)
        local_features = self.local_pointnet(local_input)
        return centroid_xyz, local_features.max(dim=1).values

class GlobalSetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, mlp_dims):
        super().__init__()
        self.pointnet = SharedPointMLP(3 + input_feature_dim, mlp_dims)
    def forward(self, xyz, features):
        local_input = torch.cat([xyz, features], dim=-1)[None]
        encoded = self.pointnet(local_input)
        return encoded.max(dim=1).values.squeeze(0)

class SmallTensorPointNetPlusPlus(nn.Module):
    def __init__(self, classes=4):
        super().__init__()
        self.sa1 = SetAbstraction(0, [8, 8, 16])
        self.sa2 = SetAbstraction(16, [16, 16, 24])
        self.sa3 = GlobalSetAbstraction(24, [24, 32, 48])
        self.classifier = nn.Sequential(nn.Linear(48, 24), nn.ReLU(), nn.Dropout(0.4), nn.Linear(24, classes))
    def forward(self, xyz):
        xyz1, features1 = self.sa1(xyz, None, centroid_count=16, radius=0.2, neighbors=8)
        xyz2, features2 = self.sa2(xyz1, features1, centroid_count=4, radius=0.4, neighbors=8)
        global_feature = self.sa3(xyz2, features2)
        return self.classifier(global_feature[None])

points = torch.rand(32, 3, device=device)
pointnet_pp = SmallTensorPointNetPlusPlus().to(device)
pointnet_logits = pointnet_pp(points)
pointnet_logits.square().mean().backward()
assert sum(isinstance(module, nn.BatchNorm1d) for module in pointnet_pp.sa1.modules()) == 3
assert sum(isinstance(module, nn.BatchNorm1d) for module in pointnet_pp.sa2.modules()) == 3
assert sum(isinstance(module, nn.BatchNorm1d) for module in pointnet_pp.sa3.modules()) == 3


## 2. DGCNN: four dynamic EdgeConv blocks


In [ ]:
def knn_indices(features, k):
    distance = torch.cdist(features, features)
    return distance.topk(k=k + 1, largest=False).indices[:, 1:]

class EdgeConvBlock(nn.Module):
    def __init__(self, input_dim, output_dim, k=4):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(nn.Linear(2 * input_dim, output_dim, bias=False), nn.BatchNorm1d(output_dim), nn.LeakyReLU(0.2))
    def forward(self, features):
        neighbor_ids = knn_indices(features.detach(), self.k)
        center = features[:, None].expand(-1, self.k, -1)
        neighbor = features[neighbor_ids]
        edge = torch.cat([center, neighbor - center], dim=-1)
        encoded = self.mlp(edge.reshape(-1, edge.size(-1))).view(features.size(0), self.k, -1)
        return encoded.max(dim=1).values

class SmallTensorDGCNN(nn.Module):
    def __init__(self, classes=4, k=4):
        super().__init__()
        self.edge1 = EdgeConvBlock(3, 8, k)
        self.edge2 = EdgeConvBlock(8, 8, k)
        self.edge3 = EdgeConvBlock(8, 16, k)
        self.edge4 = EdgeConvBlock(16, 24, k)
        self.embedding = nn.Linear(8 + 8 + 16 + 24, 32)
        self.classifier = nn.Sequential(nn.Linear(64, 32), nn.LeakyReLU(0.2), nn.Dropout(0.5), nn.Linear(32, classes))
    def forward(self, xyz):
        x1 = self.edge1(xyz)
        x2 = self.edge2(x1)
        x3 = self.edge3(x2)
        x4 = self.edge4(x3)
        local = torch.cat([x1, x2, x3, x4], dim=-1)
        embedded = F.leaky_relu(self.embedding(local), 0.2)
        global_max = embedded.max(dim=0).values
        global_mean = embedded.mean(dim=0)
        return self.classifier(torch.cat([global_max, global_mean], dim=-1)[None])

dgcnn = SmallTensorDGCNN().to(device)
dgcnn_logits = dgcnn(points)
assert sum(isinstance(module, EdgeConvBlock) for module in dgcnn.modules()) == 4


## 3. Point Transformer vector attention


In [ ]:
class PointTransformerLayer(nn.Module):
    def __init__(self, channels=16, k=4):
        super().__init__()
        self.k = k
        self.query = nn.Linear(channels, channels)
        self.key = nn.Linear(channels, channels)
        self.value = nn.Linear(channels, channels)
        self.position = nn.Sequential(nn.Linear(3, channels), nn.ReLU(), nn.Linear(channels, channels))
        self.attention = nn.Sequential(nn.Linear(channels, channels), nn.ReLU(), nn.Linear(channels, channels))
    def forward(self, xyz, features):
        neighbor_ids = knn_indices(xyz, self.k)
        relative = xyz[:, None] - xyz[neighbor_ids]
        positional = self.position(relative)
        query = self.query(features)[:, None]
        key = self.key(features)[neighbor_ids]
        value = self.value(features)[neighbor_ids]
        logits = self.attention(query - key + positional)
        weight = logits.softmax(dim=1)
        return (weight * (value + positional)).sum(dim=1)

point_features = nn.Linear(3, 16).to(device)(points)
point_transformer = PointTransformerLayer().to(device)
point_transformer_output = point_transformer(points, point_features)


## 4. PointPillars PFN: raw + cluster offset + pillar-center offset


In [ ]:
class PillarFeatureNet(nn.Module):
    def __init__(self, output_channels=16, voxel_size=0.5):
        super().__init__()
        self.output_channels = output_channels
        self.voxel_size = voxel_size
        self.point_linear = nn.Linear(9, output_channels, bias=False)
        self.point_norm = nn.BatchNorm1d(output_channels)
    def forward(self, xyz, intensity):
        raw_ids = torch.floor(xyz[:, :2] / self.voxel_size).long()
        minimum_id = raw_ids.min(dim=0).values
        local_ids = raw_ids - minimum_id
        height = int(local_ids[:, 1].max().item()) + 1
        width = int(local_ids[:, 0].max().item()) + 1
        bev = torch.zeros(self.output_channels, height, width, device=xyz.device)
        unique_ids = torch.unique(local_ids, dim=0)
        for local_id in unique_ids:
            mask = (local_ids == local_id).all(dim=1)
            pillar_xyz = xyz[mask]
            pillar_intensity = intensity[mask, None]
            cluster_offset = pillar_xyz - pillar_xyz.mean(dim=0, keepdim=True)
            raw_id = local_id + minimum_id
            center_xy = (raw_id.float() + 0.5) * self.voxel_size
            center_offset_xy = pillar_xyz[:, :2] - center_xy
            augmented = torch.cat([pillar_xyz, pillar_intensity, cluster_offset, center_offset_xy], dim=-1)
            hidden = F.relu(self.point_norm(self.point_linear(augmented)))
            pillar_feature = hidden.max(dim=0).values
            bev[:, int(local_id[1]), int(local_id[0])] = pillar_feature
        return bev

pillar_encoder = PillarFeatureNet().to(device)
pillar_seed = torch.rand(16, 3, device=device) * 2.0
lidar_points = pillar_seed.repeat_interleave(2, dim=0)
lidar_points[1::2, 2] += 0.01
intensity = torch.rand(32, device=device)
bev = pillar_encoder(lidar_points, intensity)
assert pillar_encoder.point_linear.in_features == 9


## 5. CenterPoint center heatmap + center-only box regression

Gaussian radius는 projected box size에서 계산한다. Regression heads는 offset, height, dimensions, yaw sine/cosine, planar velocity를 출력한다.


In [ ]:
class CenterPointHead(nn.Module):
    def __init__(self, input_channels=16, hidden=24, classes=2):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(input_channels, hidden, 3, padding=1), nn.BatchNorm2d(hidden), nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.BatchNorm2d(hidden), nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(hidden, classes, 1)
        self.offset = nn.Conv2d(hidden, 2, 1)
        self.height = nn.Conv2d(hidden, 1, 1)
        self.dimensions = nn.Conv2d(hidden, 3, 1)
        self.rotation = nn.Conv2d(hidden, 2, 1)
        self.velocity = nn.Conv2d(hidden, 2, 1)
        nn.init.constant_(self.heatmap.bias, -2.19)
    def forward(self, bev):
        hidden = self.backbone(bev)
        return {"heatmap_logits": self.heatmap(hidden), "offset": self.offset(hidden), "height": self.height(hidden), "dimensions": self.dimensions(hidden), "rotation": self.rotation(hidden), "velocity": self.velocity(hidden)}

def gaussian_radius(height, width, min_overlap=0.7):
    a1 = 1.0
    b1 = height + width
    c1 = width * height * (1 - min_overlap) / (1 + min_overlap)
    sq1 = math.sqrt(max(b1 * b1 - 4 * a1 * c1, 0.0))
    r1 = (b1 + sq1) / 2
    a2 = 4.0
    b2 = 2 * (height + width)
    c2 = (1 - min_overlap) * width * height
    sq2 = math.sqrt(max(b2 * b2 - 4 * a2 * c2, 0.0))
    r2 = (b2 + sq2) / 2
    a3 = 4 * min_overlap
    b3 = -2 * min_overlap * (height + width)
    c3 = (min_overlap - 1) * width * height
    sq3 = math.sqrt(max(b3 * b3 - 4 * a3 * c3, 0.0))
    r3 = (b3 + sq3) / 2
    return min(r1, r2, r3)

def draw_gaussian(heatmap, center_x, center_y, radius):
    height, width = heatmap.shape
    yy, xx = torch.meshgrid(torch.arange(height, device=heatmap.device), torch.arange(width, device=heatmap.device), indexing="ij")
    diameter = 2 * radius + 1
    sigma = max(diameter / 6.0, 1e-3)
    gaussian = torch.exp(-((xx - center_x).square() + (yy - center_y).square()) / (2 * sigma * sigma))
    return torch.maximum(heatmap, gaussian)

def modified_focal_loss(logits, target, alpha=2.0, beta=4.0):
    probability = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
    positive = target.eq(1.0)
    negative = target.lt(1.0)
    negative_weight = (1 - target).pow(beta)
    positive_loss = torch.log(probability) * (1 - probability).pow(alpha) * positive
    negative_loss = torch.log(1 - probability) * probability.pow(alpha) * negative_weight * negative
    count = positive.float().sum().clamp_min(1.0)
    return -(positive_loss.sum() + negative_loss.sum()) / count

def gather_at_centers(feature_map, centers):
    values = []
    for batch_index in range(feature_map.size(0)):
        x = centers[batch_index, :, 0]
        y = centers[batch_index, :, 1]
        values.append(feature_map[batch_index, :, y, x].transpose(0, 1))
    return torch.stack(values)

centerpoint = CenterPointHead().to(device)
prediction = centerpoint(bev[None])
center = torch.tensor([[[1, 1]]], device=device)
class_id = 1
heatmap_target = torch.zeros_like(prediction["heatmap_logits"])
box_width_cells = 2.4
box_length_cells = 3.2
radius = max(0, int(gaussian_radius(box_length_cells, box_width_cells)))
heatmap_target[0, class_id] = draw_gaussian(heatmap_target[0, class_id], center_x=1, center_y=1, radius=radius)
regression_target = {
    "offset": torch.tensor([[[0.2, 0.3]]], device=device),
    "height": torch.tensor([[[0.5]]], device=device),
    "dimensions": torch.tensor([[[1.5, 2.0, 1.2]]], device=device),
    "rotation": torch.tensor([[[0.0, 1.0]]], device=device),
    "velocity": torch.tensor([[[0.4, -0.1]]], device=device),
}
centerpoint_loss = modified_focal_loss(prediction["heatmap_logits"], heatmap_target)
for name, target in regression_target.items():
    predicted = gather_at_centers(prediction[name], center)
    centerpoint_loss = centerpoint_loss + F.l1_loss(predicted, target)
centerpoint_loss.backward()


## 6. CenterPoint local-max suppression and multi-class top-K decode


In [ ]:
def decode_centerpoint(prediction, voxel_size=0.5, k=5):
    heatmap = torch.sigmoid(prediction["heatmap_logits"])
    local_max = F.max_pool2d(heatmap, kernel_size=3, stride=1, padding=1)
    heatmap = heatmap * (local_max == heatmap)
    batch, classes, height, width = heatmap.shape
    per_class_k = min(k, height * width)
    class_scores, class_indices = torch.topk(heatmap.view(batch, classes, -1), per_class_k, dim=-1)
    final_k = min(k, classes * per_class_k)
    scores, flattened = torch.topk(class_scores.reshape(batch, -1), final_k, dim=-1)
    class_ids = torch.div(flattened, per_class_k, rounding_mode="floor")
    spatial_indices = class_indices.reshape(batch, -1).gather(1, flattened)
    y = torch.div(spatial_indices, width, rounding_mode="floor")
    x = spatial_indices % width
    centers = torch.stack([x, y], dim=-1)
    offset = gather_at_centers(prediction["offset"], centers)
    z = gather_at_centers(prediction["height"], centers)
    dimensions = gather_at_centers(prediction["dimensions"], centers)
    rotation = gather_at_centers(prediction["rotation"], centers)
    velocity = gather_at_centers(prediction["velocity"], centers)
    center_x = (x.float() + offset[..., 0]) * voxel_size
    center_y = (y.float() + offset[..., 1]) * voxel_size
    yaw = torch.atan2(rotation[..., 0], rotation[..., 1])
    boxes = torch.cat([center_x[..., None], center_y[..., None], z, dimensions, yaw[..., None], velocity], dim=-1)
    return scores, class_ids, boxes

scores, classes, boxes = decode_centerpoint(prediction, k=5)
assert boxes.size(-1) == 9
assert "velocity" in prediction
print("PointNet++ logits:", pointnet_logits.shape)
print("DGCNN logits:", dgcnn_logits.shape)
print("Point Transformer:", point_transformer_output.shape)
print("PointPillars BEV:", bev.shape)
print("CenterPoint boxes:", boxes.shape)
